# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df

import importlib
import bodaqs_analysis.widgets.event_browser as eb
importlib.reload(eb)

from bodaqs_analysis.widgets.metric_histogram_widget import make_metric_histogram_widget
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)

logging.getLogger("bodaqs_analysis").setLevel(logging.DEBUG)
logging.getLogger("bodaqs_analysis.detect").setLevel(logging.INFO)

logger = logging.getLogger(__name__)


In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-25*.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"
INGESTION_MODE = "tolerant"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock_dom_suspension [mm]": 170.0,
    "rear_shock_dom_suspension [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [ ]:
# ---- Batch orchestration ----

pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

logger.info(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    logger.info("  %s", p.name)

events_all = []
metrics_all = []
results_all = []  # optional if you want to inspect per-file outputs
sessions_by_id = {}
schema = None

for p in CSV_FILES:
    logger.info(f"Processing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
        strict = (INGESTION_MODE=="strict"),
    )
    session = results["session"]
    sid = str(session["session_id"])
    sessions_by_id[sid] = session
    #results_all.append((p, results))  # optional

    # pull from the correct keys
    ev = results["events"]
    mt = results["metrics"]
    if schema is None:
        schema = results["schema"]
    events_df = results["events"]

    # only append if non-empty (optional, but handy)
    if isinstance(ev, pd.DataFrame) and not ev.empty:
        events_all.append(ev)
    if isinstance(mt, pd.DataFrame) and not mt.empty:
        metrics_all.append(mt)

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

logging.debug("events_df: %s", events_df.shape)
logging.debug("metrics_df: %s", metrics_df.shape)






INFO:__main__:Found 13 files:
INFO:__main__:  2026-01-25_09-45-58.CSV
INFO:__main__:  2026-01-25_10-09-00.CSV
INFO:__main__:  2026-01-25_10-27-52.CSV
INFO:__main__:  2026-01-25_10-49-10.CSV
INFO:__main__:  2026-01-25_11-11-29.CSV
INFO:__main__:  2026-01-25_11-35-12.CSV
INFO:__main__:  2026-01-25_12-35-01.CSV
INFO:__main__:  2026-01-25_12-54-47.CSV
INFO:__main__:  2026-01-25_13-21-24.CSV
INFO:__main__:  2026-01-25_13-42-38.CSV
INFO:__main__:  2026-01-25_14-04-23.CSV
INFO:__main__:  2026-01-25_14-13-31.CSV
INFO:__main__:  2026-01-25_14-13-44.CSV
INFO:__main__:Processing 2026-01-25_09-45-58.CSV ...
INFO:bodaqs_analysis.pipeline:Session load complete: C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-25_09-45-58.CSV
INFO:bodaqs_analysis.pipeline:Session pre-process complete
DEBUG:bodaqs_analysis.pipeline:time_s start/end: 0.002 .. 154.954
DEBUG:bodaqs_analysis.pipeline:dt median/min/max: 0.0019999999999997797 / 0.001999999999981128 / 0.0020000000000095497
DEBUG:bodaqs_analysis.pipeline:s

In [ ]:
make_metric_histogram_widget(
    events_df,
    metrics_df,
    event_type_col="event_name",
    signal_col="signal_col",
    default_bins=10,
)


In [ ]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader

def session_loader(session_id: str) -> dict:
    return sessions_by_id[str(session_id)]

wb = make_event_browser_widget_for_loader(
    schema,
    events_df,
    session_loader=session_loader,
)


In [ ]:
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_widget

handles = make_metric_scatter_widget(
    events_df=events_df,
    metrics_df=metrics_df,
    event_type_col="event_name",   # or "event_type"/"schema_id" depending on your table
    signal_col="signal_col",
)

viz_df = handles["viz_df"]  # handy for debugging


In [ ]:
from bodaqs_analysis.widgets.signal_histogram_widget import (
    make_signal_histogram_widget_for_loader
)

def session_loader(session_id: str) -> dict:
    return sessions_by_id[str(session_id)]

wh = make_signal_histogram_widget_for_loader(
    events_df,
    session_loader=session_loader,
    default_bins=50,
)
